# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nooragab/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/nooragab/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — check repo root"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship
Starter data found. You're ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## Task Type: Ranking, implemented via Binary Classification

My lane (Refresh / Content Opportunity Scoring) is fundamentally a **ranking** problem:
given limited review capacity, order pages from "most worth reviewing" to "least worth
reviewing." I implement this ranking through **binary classification** — training a model
to output a probability that a page is "declining," then using that probability to rank
pages. This mirrors what the starter pipeline does: a classifier's predicted probability
becomes the ranking signal, and Precision@K measures whether the top of that ranking is
actually useful.

It is not clustering (I'm not grouping similar pages without a target) and not pure
regression (I don't need an exact numeric traffic forecast — I need a relative ordering of
review priority).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Task type check: how many pages fall into each trend bucket "
      f"(the raw material for the ranking signal)?")
print(df["trend_direction"].value_counts())


Task type check: how many pages fall into each trend bucket (the raw material for the ranking signal)?
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## Target / Proxy

**Starter proxy target:** `is_declining_label = (trend_direction == "down")` — a binary
label based on the page's *current* trend bucket.

**Why this is a proxy, not the ideal target:** this label is calculated from the current
window, not a future observed outcome — the lane guide flags this exact weakness as a
"beginner proxy," useful to prove the workflow end-to-end but not the strongest possible
target.

**A stronger target for later weeks:** `features from a prior 90-day window -> decline
over the next 30 days`. This shifts the label to something genuinely unknown at prediction
time, removing the risk of the model just re-describing the present instead of forecasting.

For now, I use the starter proxy label to keep things concrete and buildable.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print("Target distribution:")
print(df["is_declining_label"].value_counts(normalize=True).round(3))
df[["content_id", "trend_direction", "is_declining_label"]].head(10)

Target distribution:
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


,content_id,trend_direction,is_declining_label
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
6,content_9a34b442b552,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## Success Metric

**Primary metric: Precision@50** — matching a realistic weekly editor review capacity.

**Why this metric and not plain accuracy:** an editor only acts on a short list.
Precision@K directly answers "of the pages I actually have time to review, how many were
worth reviewing?" Plain accuracy would reward correctly labeling the majority-class pages
that aren't candidates at all — not the decision actually being made.

**Baseline to beat:** the starter's hand-written rule reached Precision@50 = 0.240. Any
model built in this lane needs to clear that bar to justify the added complexity.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def precision_at_k(scores, labels, k):
    import numpy as np
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Sanity-check the metric function on the label itself (a perfect "oracle" ranking)
perfect_score = df["is_declining_label"]  # ranking by the label itself = best possible case
oracle_precision = precision_at_k(perfect_score, df["is_declining_label"], 50)
print(f"Oracle (perfect) Precision@50 on this data: {oracle_precision:.3f}")
print(f"Baseline rule (from notebook 01) reached: 0.240")
print(f"Bar to beat with my model: 0.240")

Oracle (perfect) Precision@50 on this data: 1.000
Baseline rule (from notebook 01) reached: 0.240
Bar to beat with my model: 0.240


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## The Unit of Analysis

One row = one page (`content_id`), observed over its trailing 90-day window. Below is a
real slice of the dataframe showing exactly what one row of my lane's data looks like.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

unit_of_analysis = df[[
    "content_id", "client_id", "impressions_90d", "clicks_90d",
    "avg_position", "ctr", "days_since_last_update",
    "content_age_days", "word_count", "trend_direction", "is_declining_label"
]]

print(f"Grain check — unique content_id: {df['content_id'].nunique():,} "
      f"vs total rows: {len(df):,} (should match: one row per page)")
unit_of_analysis.head(5)

Grain check — unique content_id: 30,000 vs total rows: 30,000 (should match: one row per page)


,content_id,client_id,impressions_90d,clicks_90d,avg_position,ctr,days_since_last_update,content_age_days,word_count,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,29,10.6,0.76,20,187,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,20.3,0.05,25,445,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,36.5,0.09,20,141,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,11751,58,6.2,0.49,22,463,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,44.0,0.13,14,263,2803.0,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## Why ML Beats a Fixed Rule Here

A fixed rule (e.g. "stale AND visible") can only combine a handful of thresholds an editor
can hold in their head. This lane has many observable signals — impressions, clicks,
position, freshness, word count, CTR, engagement — and how they interact to predict decline
is tangled, not a clean straight line.

This isn't a guess — I confirmed it directly on this data. The hand-written rule
(stale × visible × impressions, same formula as notebook 02) reached Precision@50 = 0.680
here, while the starter pipeline's random forest — trained on the full feature set —
reached 0.740. Even a well-tuned hand rule tops out with a narrow set of signals; the model's
edge comes from combining many weak signals the rule can't hold at once. Different rule
formulations give different baselines (the pipeline's weighted-score rule reached only 0.240)
— which itself shows how sensitive fixed rules are to their exact design, unlike a model that
learns the combination automatically.

A plain rule stays useful as a transparent baseline and as reason-code logic in the final
queue — but it isn't the ranking engine.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Recreate the hand-written rule from notebook 02, computed live on this data
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
hand_rule_score = stale * visible * df["impressions_90d"]

import numpy as np
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

rule_p50 = precision_at_k(hand_rule_score, df["is_declining_label"], 50)
print(f"Hand-written rule Precision@50 (computed live on this data): {rule_p50:.3f}")
print(f"\nFor comparison, the starter pipeline's trained random forest reached: 0.740")
print(f"Model improvement over the rule: {0.740/rule_p50:.1f}x")

Hand-written rule Precision@50 (computed live on this data): 0.680

For comparison, the starter pipeline's trained random forest reached: 0.740
Model improvement over the rule: 1.1x


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.